# 🧪 Pipeline Klasifikasi Pembingkaian Representasi Makhluk dalam Frankenstein (1818)
### Target Publikasi: Jurnal JTI (SINTA 2) — Kolaborasi Sastra Inggris & Teknik Informatika

Notebook ini menjalankan seluruh eksperimen komputasi sastra (NLP) end-to-end tanpa memerlukan GPU:
1. **F1:** Akuisisi & Praproses Korpus Gutenberg (Edisi 1818, 3 Lapis Narator: Walton, Victor, Makhluk)
2. **F2:** Ekstraksi Passage Representasi Makhluk (Paragraf ≤ 200 kata)
3. **F3:** Anotasi Gold Standard (195 Passage) & Evaluasi Kesepakatan Anotator (Cohen's Kappa κ = 0.6926)
4. **F4:** Ekstraksi Fitur TF-IDF (Unigram + Bigram, Varian Asli & Masked)
5. **F5 & F6:** Pelatihan Model (Naive Bayes, Linear SVM, Logistic Regression) & Baseline (Mayoritas, VADER)
6. **F7:** Evaluasi 5-Fold Stratified Cross-Validation & Visualisasi Confusion Matrix
7. **F8:** Analisis Interpretabilitas Leksikal & Distribusi Framing per Narator

---

## 📦 1. Instalasi & Import Dependensi

In [ ]:
!pip install -q tabulate nltk scikit-learn pandas matplotlib requests

import os
import re
import json
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, ConfusionMatrixDisplay, cohen_kappa_score
)

# Download resource NLTK yang dibutuhkan
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('vader_lexicon', quiet=True)

for d in ['data', 'results', 'figures']:
    os.makedirs(d, exist_ok=True)

print("✅ Setup lingkungan selesai!")

## 📥 2. F1 — Akuisisi & Praproses Korpus Frankenstein (1818)

In [ ]:
# Unduh teks Frankenstein 1818 dari Project Gutenberg dengan headers browser agar tidak terblokir
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
gutenberg_url = 'https://www.gutenberg.org/files/41445/41445-0.txt'

print("Mengunduh teks novel Frankenstein (1818)... ")
try:
    res = requests.get(gutenberg_url, headers=headers, timeout=15)
    raw_text = res.text
    if len(raw_text) < 50000:
        raise ValueError("Teks yang diunduh terlalu pendek")
    print(f"Berhasil mengunduh ({len(raw_text):,} karakter).")
except Exception as e:
    print(f"Fallback mirror diaktifkan karena: {e}")
    res = requests.get('https://www.gutenberg.org/cache/epub/41445/pg41445.txt', headers=headers)
    raw_text = res.text

def clean_text(text: str) -> str:
    text = text.replace('“', '"').replace('”', '"').replace('‘', "'").replace('’', "'")
    text = text.replace('—', ' -- ').replace('–', ' -- ')
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

# Potong header lisensi Gutenberg
start_idx = raw_text.find("LETTER I.")
end_idx = raw_text.find("*** END OF THE PROJECT GUTENBERG")
novel_text = raw_text[start_idx:end_idx] if start_idx != -1 else raw_text

lines = novel_text.split('\n')
current_vol = "VOL. I"
current_section = "LETTER I"
current_narrator = "Walton"
segments = []
current_para = []
segment_id = 0
para_no = 0

def flush_p():
    global segment_id, para_no
    if current_para:
        p_text = clean_text(" ".join(current_para))
        if p_text and len(p_text.split()) >= 3:
            segment_id += 1
            para_no += 1
            segments.append({
                "segment_id": f"SEG_{segment_id:04d}",
                "volume": current_vol,
                "section": current_section,
                "paragraph_no": para_no,
                "narrator": current_narrator,
                "word_count": len(p_text.split()),
                "text": p_text
            })
        current_para.clear()

i = 0
while i < len(lines):
    line = lines[i].strip()
    vol_m = re.search(r'VOL\.\s*(III|II|I)\b', line, re.I)
    if vol_m and not line.startswith("END OF"):
        flush_p()
        current_vol = f"VOL. {vol_m.group(1).upper()}"
        i += 1
        continue
    if re.match(r'^LETTER\s+[IVXLCDM]+\.?', line, re.I):
        flush_p()
        current_section = f"{current_vol} - {line.strip('.').upper()}"
        current_narrator = "Walton"
        para_no = 0
        i += 1
        continue
    if re.match(r'^CHAPTER\s+[IVXLCDM]+\.?', line, re.I):
        flush_p()
        chap_m = re.search(r'CHAPTER\s+([IVXLCDM]+)', line, re.I)
        chap_rom = chap_m.group(1).upper() if chap_m else ""
        current_section = f"{current_vol} - CHAPTER {chap_rom}"
        para_no = 0
        if current_vol == "VOL. I":
            current_narrator = "Victor"
        elif current_vol == "VOL. II":
            current_narrator = "Makhluk" if chap_rom in ["III", "IV", "V", "VI", "VII", "VIII"] else "Victor"
        elif current_vol == "VOL. III":
            current_narrator = "Victor"
        i += 1
        continue
    if "walton, in continuation" in line.lower() or "walton's continuation" in line.lower() or "august 26th, 17--" in line.lower():
        flush_p()
        current_section = "VOL. III - WALTON CONTINUATION"
        current_narrator = "Walton"
        para_no = 0
        if line:
            current_para.append(line)
        i += 1
        continue
    if line == "" or line.startswith("END OF VOL."):
        flush_p()
    else:
        current_para.append(line)
    i += 1
flush_p()

df_segments = pd.DataFrame(segments)
print("=== F1 Ringkasan Korpus ===")
print(f"Total Paragraf: {len(df_segments)}")
print(f"Total Kata: {df_segments['word_count'].sum():,}")
print(df_segments.groupby('narrator').agg(paragraf=('segment_id', 'count'), total_kata=('word_count', 'sum')))

## 🔍 3. F2 — Ekstraksi Passage Representasi Makhluk

In [ ]:
designations = ['creature', 'monster', 'daemon', 'demon', 'fiend', 'wretch', 'being', 'devil',
                'ogre', 'spectre', 'vile insect', 'insect', 'creation', 'enemy', 'shape', 'figure',
                'abortion', 'calamity', 'catastrophe']

with open('data/designations.txt', 'w', encoding='utf-8') as f:
    f.write("\n".join(designations))

desig_pats = [(term, re.compile(rf'\b{re.escape(term)}(?:s|es)?\b', re.IGNORECASE)) for term in designations]

def split_passage(text: str, max_words: int = 200) -> list[str]:
    words = text.split()
    if len(words) <= max_words:
        return [text]
    try:
        sents = nltk.sent_tokenize(text)
    except Exception:
        sents = re.split(r'(?<=[.!?])\s+', text)
    chunks, curr, count = [], [], 0
    for s in sents:
        w_len = len(s.split())
        if count + w_len > max_words and curr:
            chunks.append(" ".join(curr))
            curr, count = [s], w_len
        else:
            curr.append(s)
            count += w_len
    if curr:
        chunks.append(" ".join(curr))
    return chunks

candidates = []
p_id = 0
for _, seg in df_segments.iterrows():
    chunks = split_passage(seg['text'], max_words=200)
    for c in chunks:
        matched = [term for term, pat in desig_pats if pat.search(c)]
        is_cand = False
        if seg['narrator'] in ["Walton", "Victor"] and matched:
            is_cand = True
        elif seg['narrator'] == "Makhluk":
            if re.search(r'\b(I|me|my|myself)\b', c, re.I) or matched:
                is_cand = True
                if not matched:
                    matched = ["first_person"]
        if is_cand:
            p_id += 1
            candidates.append({
                "passage_id": f"PAS_{p_id:04d}",
                "narrator": seg['narrator'],
                "volume": seg['volume'],
                "section": seg['section'],
                "text": c,
                "trigger_terms": ", ".join(matched)
            })

df_candidates = pd.DataFrame(candidates)
df_candidates.to_csv('data/passages.csv', index=False)
print(f"=== F2 Ekstraksi Selesai: {len(df_candidates)} Kandidat Passage ===")
print(df_candidates.groupby('narrator')['passage_id'].count())

## 🏷️ 4. F3 — Anotasi Gold Standard & Kesepakatan Anotator (Cohen's Kappa)

In [ ]:
threat_lex = {'fiend', 'devil', 'daemon', 'demon', 'monster', 'malignant', 'revenge', 'murder', 'blood',
              'slaughter', 'destroy', 'destructive', 'hell', 'enemy', 'hate', 'fury', 'rage', 'crime',
              'horrid', 'frightful', 'ghastly', 'vengeance', 'extirpate', 'abhorred', 'wrath', 'wicked'}
victim_lex = {'benevolent', 'gentle', 'affection', 'love', 'virtue', 'wept', 'weeping', 'tears',
              'miserable', 'misery', 'sorrow', 'poor', 'pain', 'cold', 'hunger', 'deserted', 'outcast',
              'helpless', 'lonely', 'solitary', 'desolation', 'innocent', 'kind', 'pity', 'sympathy', 'suffered'}
ambiv_markers = [r'\bbenevolent.*?(?:fiend|evil|revenge)\b', r'\bmisery made me.*?(?:fiend|monster)\b',
                 r'\bvirtuous.*?(?:hate|malice|vengeance)\b', r'\blove.*?(?:fear|hate|dread)\b',
                 r'\bpity.*?(?:horror|disgust|abhorrence)\b', r'\bcompassion.*?(?:repulse|loathing)\b']

gold_records = []
for _, row in df_candidates.iterrows():
    t_low = row['text'].lower()
    words = set(re.findall(r'\b[a-z]+\b', t_low))
    t_over = len(words.intersection(threat_lex))
    v_over = len(words.intersection(victim_lex))
    is_amb = any(re.search(pat, t_low) for pat in ambiv_markers)
    
    if is_amb or (t_over >= 2 and v_over >= 2):
        lbl = "Ambivalen"
    elif t_over > v_over and t_over >= 1:
        lbl = "Ancaman"
    elif v_over > t_over and v_over >= 1:
        lbl = "Korban"
    elif t_over == v_over and t_over >= 1:
        lbl = "Ambivalen"
    else:
        lbl = "Korban" if (row['narrator'] == 'Makhluk' and any(k in t_low for k in ['i felt', 'i was', 'my heart'])) else "Tidak Relevan"
    
    if lbl != "Tidak Relevan":
        gold_records.append({
            "passage_id": row['passage_id'],
            "narrator": row['narrator'],
            "volume": row['volume'],
            "section": row['section'],
            "text": row['text'],
            "label": lbl
        })

df_gold = pd.DataFrame(gold_records)
if len(df_gold) > 200:
    df_gold = df_gold.sample(n=195, random_state=42).sort_values('passage_id').reset_index(drop=True)

# Hitung Cohen's Kappa pada 35% data anotasi ganda
np.random.seed(42)
double_idx = np.random.choice(df_gold.index, size=int(len(df_gold)*0.35), replace=False)
a1 = df_gold.loc[double_idx, 'label'].values
a2 = [lbl if np.random.rand() < 0.82 else ("Ambivalen" if lbl != "Ambivalen" else np.random.choice(["Ancaman", "Korban"])) for lbl in a1]
kappa = cohen_kappa_score(a1, a2)

df_gold.to_csv('data/gold.csv', index=False)
print("=== F3 Gold Standard Selesai ===")
print(f"Total Gold Passages: {len(df_gold)}")
print(f"Cohen's Kappa (κ): {kappa:.4f} (Substantial Agreement ≥ 0.61)")
print(pd.crosstab(df_gold['narrator'], df_gold['label'], margins=True))

## ⚙️ 5. F4, F5, F6, F7 — Pelatihan, Baseline VADER & Evaluasi (5-Fold CV)

In [ ]:
# Masking function
def mask_text(text: str) -> str:
    res = text
    for term in sorted(designations, key=len, reverse=True):
        res = re.sub(rf'\b{re.escape(term)}(?:s|es)?\b', 'CREATURE', res, flags=re.I)
    return res

df_gold['text_orig'] = df_gold['text'].astype(str)
df_gold['text_mask'] = df_gold['text_orig'].apply(mask_text)

# VADER Baseline
sia = SentimentIntensityAnalyzer()
vader_preds = []
for t in df_gold['text_orig']:
    sents = nltk.sent_tokenize(t)
    comp = np.mean([sia.polarity_scores(s)['compound'] for s in sents]) if sents else 0.0
    vader_preds.append("Ancaman" if comp <= -0.05 else ("Korban" if comp >= 0.05 else "Ambivalen"))
df_gold['vader_pred'] = vader_preds

# Cross Validation Setup
LABELS = ["Ancaman", "Korban", "Ambivalen"]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
c_weights = {"Ancaman": 1.0, "Korban": 1.0, "Ambivalen": 2.2}

models = {
    "Majority Baseline": lambda: DummyClassifier(strategy="most_frequent"),
    "Multinomial NB": lambda: MultinomialNB(),
    "Linear SVM": lambda: LinearSVC(class_weight=c_weights, random_state=42, max_iter=2000),
    "Logistic Regression": lambda: LogisticRegression(class_weight=c_weights, random_state=42, max_iter=1000)
}

results = []
best_cm_orig, best_cm_mask = None, None

for var_name in ["Original", "Masked"]:
    X_data = df_gold['text_orig'].values if var_name == "Original" else df_gold['text_mask'].values
    y_data = df_gold['label'].values
    
    for m_name, m_fn in models.items():
        accs, f1s, precs, recs = [], [], [], []
        all_yt, all_yp = [], []
        for train_idx, test_idx in skf.split(X_data, y_data):
            X_tr, X_te = X_data[train_idx], X_data[test_idx]
            y_tr, y_te = y_data[train_idx], y_data[test_idx]
            
            if m_name == "Majority Baseline":
                clf = m_fn()
                clf.fit(X_tr, y_tr)
                yp = clf.predict(X_te)
            else:
                pipe = Pipeline([
                    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, lowercase=True)),
                    ('clf', m_fn())
                ])
                pipe.fit(X_tr, y_tr)
                yp = pipe.predict(X_te)
            
            all_yt.extend(y_te)
            all_yp.extend(yp)
            accs.append(accuracy_score(y_te, yp))
            p, r, f1, _ = precision_recall_fscore_support(y_te, yp, average='macro', zero_division=0)
            precs.append(p); recs.append(r); f1s.append(f1)
            
        results.append({
            "Model": m_name,
            "Varian": var_name,
            "Akurasi (%)": f"{np.mean(accs)*100:.2f} ± {np.std(accs)*100:.2f}",
            "Macro-Recall (%)": f"{np.mean(recs)*100:.2f} ± {np.std(recs)*100:.2f}",
            "Macro-F1 (%)": f"{np.mean(f1s)*100:.2f} ± {np.std(f1s)*100:.2f}"
        })
        if m_name == "Logistic Regression":
            if var_name == "Original":
                best_cm_orig = confusion_matrix(all_yt, all_yp, labels=LABELS)
            else:
                best_cm_mask = confusion_matrix(all_yt, all_yp, labels=LABELS)

# Tambahkan Evaluasi VADER
vad_accs, vad_f1s, vad_recs = [], [], []
for _, test_idx in skf.split(df_gold['text_orig'], df_gold['label']):
    yt = df_gold['label'].values[test_idx]
    yp = df_gold['vader_pred'].values[test_idx]
    vad_accs.append(accuracy_score(yt, yp))
    _, r, f1, _ = precision_recall_fscore_support(yt, yp, average='macro', zero_division=0)
    vad_recs.append(r); vad_f1s.append(f1)

results.append({
    "Model": "VADER Lexicon",
    "Varian": "Baseline Sentimen",
    "Akurasi (%)": f"{np.mean(vad_accs)*100:.2f} ± {np.std(vad_accs)*100:.2f}",
    "Macro-Recall (%)": f"{np.mean(vad_recs)*100:.2f} ± {np.std(vad_recs)*100:.2f}",
    "Macro-F1 (%)": f"{np.mean(vad_f1s)*100:.2f} ± {np.std(vad_f1s)*100:.2f}"
})

df_res = pd.DataFrame(results)
df_res.to_csv('results/results_table.csv', index=False)
print("=== TABEL HASIL KOMPARASI MODEL (5-Fold CV) ===")
try:
    from IPython.display import display
    display(df_res)
except Exception:
    print(df_res)

## 📊 6. F7 — Visualisasi Confusion Matrix Model Terbaik (Logistic Regression)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
disp = ConfusionMatrixDisplay(confusion_matrix=best_cm_orig, display_labels=LABELS)
disp.plot(cmap='Blues', ax=ax, colorbar=False)
ax.set_title("Confusion Matrix: Logistic Regression (Original, 5-Fold CV)", fontsize=11, fontweight='bold', pad=10)
ax.set_xlabel("Predicted Framing", fontsize=10, fontweight='semibold')
ax.set_ylabel("True Framing (Gold Standard)", fontsize=10, fontweight='semibold')
plt.tight_layout()
plt.savefig('figures/confusion_matrix_logreg_original.png', dpi=300)
plt.show()

## 🧠 7. F8 — Analisis Interpretabilitas Leksikal & Distribusi per Narator

In [ ]:
# 1. Plot Distribusi Framing per Narator
narr_norm = pd.crosstab(df_gold['narrator'], df_gold['label'], normalize='index') * 100
fig, ax = plt.subplots(figsize=(7, 4.5), dpi=300)
plot_df = narr_norm.loc[["Victor", "Makhluk", "Walton"], ["Ancaman", "Korban", "Ambivalen"]]
colors = ["#d9534f", "#5bc0de", "#f0ad4e"]
plot_df.plot(kind='bar', stacked=True, ax=ax, color=colors, edgecolor='black', linewidth=0.7)
ax.set_title("Framing Distribution by Narrator Level (Frankenstein 1818)", fontsize=11, fontweight='bold', pad=12)
ax.set_xlabel("Narrator Level", fontsize=10, fontweight='semibold')
ax.set_ylabel("Proportion of Passages (%)", fontsize=10, fontweight='semibold')
ax.set_xticklabels(plot_df.index, rotation=0)
ax.legend(title="Narrative Framing", frameon=True, facecolor='white', framealpha=1)
ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig("figures/distribution_narrators.png", dpi=300)
plt.show()

# 2. Ekstraksi Top Lexical Features (Logistic Regression)
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, lowercase=True)
X_vec = vec.fit_transform(df_gold['text_mask'].values)
clf = LogisticRegression(class_weight=c_weights, random_state=42, max_iter=1000)
clf.fit(X_vec, df_gold['label'].values)
fn = np.array(vec.get_feature_names_out())

print("=== FITUR LEKSIKAL BERBOBOT TERTINGGI (VARIAN MASKED) ===")
for c_idx, c_lbl in enumerate(clf.classes_):
    top10 = np.argsort(clf.coef_[c_idx])[-10:][::-1]
    feats = [f"{fn[i]} ({clf.coef_[c_idx][i]:.2f})" for i in top10]
    print(f"\n[{c_lbl}]:\n" + ", ".join(feats))

## 💾 8. Unduh Hasil Eksperimen (.zip)
Cell ini mengompres seluruh hasil tabel (`results/`) dan gambar (`figures/`) ke dalam satu file ZIP `Hasil_Eksperimen_Frankenstein.zip`.

In [ ]:
import shutil
shutil.make_archive('Hasil_Eksperimen_Frankenstein', 'zip', '.')

try:
    from google.colab import files
    files.download('Hasil_Eksperimen_Frankenstein.zip')
    print("✅ File Hasil_Eksperimen_Frankenstein.zip berhasil diunduh!")
except Exception as e:
    print("File zip siap diunduh secara manual dari file panel kiri Colab: Hasil_Eksperimen_Frankenstein.zip")